In [ ]:
#1. pydantic
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str= Field(description="The title of the movie")
    year:int= Field(description="The year of the movie")
    director:str = Field(description="Director of the movie")

In [11]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.11'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A5FE1BD580>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A5FE4074D0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [12]:
model_with_structure.invoke("Proview detail about movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan')

In [13]:
# Message output alongside passed structure
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

In [15]:
response = model_with_structure.invoke("Provide the detail about inception movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the Inception movie. We can call function Movie with appropriate data. Provide director, rating, title, year. Inception directed by Christopher Nolan, year 2010, rating maybe 8.8 (IMDb). Provide title "Inception". Let\'s call function.', 'tool_calls': [{'id': 'fc_f16f58a2-4984-4694-9d27-2156982872dd', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 163, 'total_tokens': 274, 'completion_time': 0.232595947, 'completion_tokens_details': {'reasoning_tokens': 59}, 'prompt_time': 0.009918187, 'prompt_tokens_details': None, 'queue_time': 0.28770934, 'total_time': 0.242514134}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': N

In [17]:
# Nested structure in pydantic
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide the detail about inception movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide the detail about inception movie". Likely they want details like title, year, genres, cast, budget, etc. We have a function MovieDetails that expects fields: budget?, cast (array of name, role), genres, title, year. We can call the function to get details. Let\'s call it with appropriate parameters (maybe we leave budget null?). We\'ll fill known data: Title "Inception", Year 2010, Genres ["Action", "Adventure", "Sci-Fi"], Cast list: Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page (now Elliot Page) as Ariadne, Tom Hardy as Eames, Ken Watanabe as Saito, Marion Cotillard as Mal, Michael Caine as Professor Stephen Miles. Budget: $160 million. Let\'s call function.', 'tool_calls': [{'id': 'fc_4e608cc0-01f5-4d9e-87ac-75da296a607c', 'function': {'arguments': '{"budget":160000000,"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"A

In [ ]:
#2. typedDict
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)

model_withtypedict.invoke("Please provide the details of the movie avengers")



{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [19]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide the detail about inception movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide the detail about inception movie". Likely they want details like title, year, genres, cast, budget etc. We have a function MovieDetails that expects fields: budget (number), cast (list of dicts with name, role), genres (list of strings), title (string), year (number). We need to fill with details about Inception.\n\nWe need to call the function with appropriate data. Provide maybe budget $160 million, year 2010, genres: Action, Adventure, Sci-Fi, etc. Cast: Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page (now Elliot Page) as Ariadne, Tom Hardy as Eames, Ken Watanabe as Saito, Marion Cotillard as Mal, Michael Caine as Miles, etc. Provide maybe top cast.\n\nWe need to call function. Let\'s construct.\n\nWe\'ll call functions.MovieDetails with appropriate data.', 'tool_calls': [{'id': 'fc_a225c50d-6c6d-4eb9-ac8d-c988024aa5ef', 'function': {'arguments': '{"budget":160000

In [22]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [ ]:
# pydantic
import os
groq_api_key = os.getenv("GROQ_API_KEY")

# 3. DataClasses

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [33]:
# 3. typedDict
import os
groq_api_key = os.getenv("GROQ_API_KEY")

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str 
    email: str
    phone: str 

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [35]:
# 3. Dataclasses

import os
groq_api_key = os.getenv("GROQ_API_KEY")

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass 
class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str 
    email: str
    phone: str 

model = ChatGroq(
    model="openai/gpt-oss-120b",
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}
